# Failure Analysis

This notebook treats failures as learning assets. Instead of hiding bad outcomes, we extract them, classify them, and inspect the traces that explain what went wrong.

## Learning goals

- Recognize common failure modes in agent systems.
- Build a simple failure taxonomy.
- Inspect execution traces from failed or weak runs.
- Turn failure patterns into concrete improvement ideas.


## Concept explanation

Notebook debugging is only useful if you know which environment generated the artifacts. As before, we begin by printing the interpreter path so trace and evaluation outputs are tied to the intended uv kernel.


In [ ]:
import sys
print(sys.executable)


This setup cell imports the failure-analysis helpers from `src/` and regenerates the evaluation artifacts on demand. That keeps the notebook self-contained when it is run top to bottom on a fresh machine.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.config import get_paths
from src.evaluator import attach_failure_improvements, extract_failure_cases, run_evaluation_suite
from src.utils import display_trace, read_json

pd.set_option('display.max_colwidth', 140)
paths = get_paths()
results, summary = run_evaluation_suite(repeats=1, persist_outputs=True)


### Failure modes in agent systems

Agent workflows fail for different reasons than plain QA systems. Some failures come from retrieval, some from planning, some from tools, and some from synthesis or verification. Naming those modes helps us debug systematically.

### Failure taxonomy

The project uses a compact taxonomy with examples such as `retrieval_miss`, `bad_plan`, `tool_error`, and `ungrounded_answer`. The exact labels are less important than having a stable way to group similar problems.

## Implementation


In [ ]:
taxonomy = pd.DataFrame(
    [
        {'failure_type': 'retrieval_miss', 'meaning': 'The right evidence never entered the context window.'},
        {'failure_type': 'bad_plan', 'meaning': 'The workflow chose an unhelpful reasoning template.'},
        {'failure_type': 'tool_error', 'meaning': 'A requested tool failed or produced unusable output.'},
        {'failure_type': 'ungrounded_answer', 'meaning': 'The answer included claims the evidence could not support.'},
    ]
)
taxonomy


### Extract failure cases

Now we pull the actual failures from the evaluation results and attach suggested improvement ideas. This gives us a working queue of issues instead of a vague sense that the system sometimes struggles.


In [ ]:
failures = attach_failure_improvements(extract_failure_cases(results))
failures[['system', 'question_id', 'question', 'failure_type', 'improvement_idea']].head(12)


### Trace inspection

A single trace often explains more than a dozen summary statistics. We load one recorded workflow trace from `artifacts/traces/` and render it as a table showing the node names, inputs, and outputs.


In [ ]:
trace_files = sorted(paths.traces_dir.glob('*.json'))
example_trace_path = trace_files[0]
example_trace = read_json(example_trace_path)['trace']
print(example_trace_path.name)
display_trace(example_trace)


## Experiment

A practical improvement exercise is to count which failure types happen most often. That helps you decide whether to invest next in retrieval quality, planning logic, or verifier thresholds.


In [ ]:
experiment_frame = (
    failures.groupby(['system', 'failure_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['count', 'failure_type'], ascending=[False, True])
)
experiment_frame


## Result analysis

The goal is not to eliminate every failure in one pass. The goal is to make failure patterns visible enough that each next improvement is deliberate, testable, and easy to explain.


In [ ]:
summary


## Takeaways

- Failure analysis makes the project feel like a real research workflow instead of a static demo.
- Traces connect symptoms to concrete workflow decisions.
- Improvement ideas are most useful when tied to a named failure type.
- This notebook closes the loop between architecture, evaluation, and debugging.
